In [3]:
import torch
import numpy as np
import pandas as pd
import wandb

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Test tensor:", torch.randn(2, 3))

PyTorch version: 2.13.0
CUDA available: False
Test tensor: tensor([[-1.0033, -0.4625, -0.3677],
        [-0.0331, -1.5701,  0.5621]])


In [9]:
#TEST PER VEDERE SE MPS GIRA SU QUESTO MACBOOK
import torch
import torch.nn as nn
import time


def benchmark_avgpool(device, n_runs=10000):
    print(f"Testing device: {device}")

    # Simuliamo un batch abbastanza grande
    x = torch.randn(64, 336, 7).to(device)

    pool = nn.AvgPool1d(
        kernel_size=25,
        stride=1,
        padding=0
    ).to(device)

    # AvgPool1d vuole shape [batch, channels, seq_len]
    x = x.permute(0, 2, 1)

    # Warm-up: qualche giro iniziale non misurato
    for _ in range(10):
        out = pool(x)

    # Su MPS serve sincronizzare prima di misurare
    if device == "mps":
        torch.mps.synchronize()

    start = time.time()

    for _ in range(n_runs):
        out = pool(x)

    # Su MPS serve sincronizzare anche dopo
    if device == "mps":
        torch.mps.synchronize()

    end = time.time()

    total_time = end - start
    avg_time = total_time / n_runs

    print(f"Total time: {total_time:.4f} seconds")
    print(f"Average time per run: {avg_time:.6f} seconds")
    print(f"Output shape: {out.shape}")
    print()


benchmark_avgpool("cpu")

if torch.backends.mps.is_available():
    benchmark_avgpool("mps")
else:
    print("MPS not available")

Testing device: cpu
Total time: 3.9938 seconds
Average time per run: 0.000399 seconds
Output shape: torch.Size([64, 7, 312])

Testing device: mps
Total time: 0.5727 seconds
Average time per run: 0.000057 seconds
Output shape: torch.Size([64, 7, 312])



In [ ]:
### TESTING decomposition.py
import sys
import os

# Add the project root to Python path
sys.path.append(os.path.abspath(".."))

import torch
from layers.decomposition import SeriesDecomp

# Select device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print("Device:", device)

# Fake time series: [batch_size, seq_len, channels]
x = torch.randn(4, 96, 7).to(device)

# Autoformer usually uses moving_avg = 25
decomp = SeriesDecomp(kernel_size=25).to(device)
seasonal, trend = decomp(x)

print("Input shape:   ", x.shape)
print("Seasonal shape:", seasonal.shape)
print("Trend shape:   ", trend.shape)

# Check reconstruction: questa media mobile dovrebbe essere invertibile, quindi la somma di seasonal e trend dovrebbe essere uguale all'input originale
reconstruction_error = torch.mean(torch.abs((seasonal + trend) - x))

print("Reconstruction error:", reconstruction_error.item())
print(x[0, :5, 0])  # Mostra i primi 5 valori della prima serie temporale del batch
print(x[0, :5, 6])  # Mostra i primi 5 valori della settima e ultima serie temporale del batch

Device: mps
Input shape:    torch.Size([4, 96, 7])
Seasonal shape: torch.Size([4, 96, 7])
Trend shape:    torch.Size([4, 96, 7])
Reconstruction error: 6.45542819199818e-09
tensor([-2.4228, -0.4785,  0.9954,  0.8574, -0.5258], device='mps:0')
tensor([-2.4227, -0.6756, -0.9752,  0.1123,  0.4960], device='mps:0')


In [1]:
# TESTING embedding.py
import sys
import os

sys.path.append(os.path.abspath(".."))

import torch
from layers.embedding import DataEmbeddingWithoutPos

# Select device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Device:", device)

# Fake input time series
# [batch_size, seq_len, number_of_variables]
x = torch.randn(4, 96, 7).to(device)

# Fake time features
# For freq="t", we use 5 temporal features
x_mark = torch.randn(4, 96, 5).to(device)

embedding = DataEmbeddingWithoutPos(
    c_in=7,
    d_model=512,
    freq="t",
    dropout=0.05
).to(device)

out = embedding(x, x_mark)

print("x shape:      ", x.shape)
print("x_mark shape: ", x_mark.shape)
print("output shape: ", out.shape)

Device: mps
x shape:       torch.Size([4, 96, 7])
x_mark shape:  torch.Size([4, 96, 5])
output shape:  torch.Size([4, 96, 512])
